# PjBL – RA1: Ingestão e Preparação dos Dados
## GTA V – Catálogo de Veículos (`vehicles.json` + `vehicleHandlings.json`)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ODevAbreu/DataAnalytics/blob/main/notebooks/RA1_Ingestao_Preparacao_GTAV.ipynb)

**Fluxo do RA1:** Fontes de Dados → Ingestão → Compreensão → Avaliação da Qualidade → Limpeza e Preparação → Dados Preparados

| Etapa | Seção |
|---|---|
| Definição do problema e perguntas analíticas | 1 |
| Fontes de dados | 2 |
| Ingestão (reprodutível, commit fixado) | 3 |
| Compreensão e avaliação da qualidade | 4 |
| Limpeza e preparação | 5 |
| Conjunto de dados resultante | 6 |

> Ferramenta de processamento: **PySpark** (executado no Google Colab).
> Para reproduzir: `Ambiente de execução → Executar tudo`. Nenhum arquivo precisa ser enviado manualmente.

## 1. Definição do problema e perguntas analíticas

**Contexto.** GTA V possui um catálogo com centenas de veículos (carros, motos, aeronaves, barcos) adicionados ao longo de ~10 anos de atualizações (DLCs). Cada veículo tem atributos de desempenho (velocidade máxima, aceleração, frenagem, tração), atributos físicos de *handling* (massa, força do motor, marchas, freio) e um valor monetário no jogo.

**Problema.** Para jogadores que escolhem em qual veículo investir, e para quem estuda balanceamento de jogos, não é claro **o que determina o desempenho e o valor de um veículo**, nem se os veículos lançados nas atualizações mais recentes são sistematicamente melhores (*power creep*).

**Perguntas analíticas** (respondidas nos próximos RAs; o RA1 prepara os dados para elas):

1. **P1 – Desempenho por categoria:** como velocidade máxima, aceleração e frenagem variam entre **classes** (SUPER, SPORT, MUSCLE...) e **tipos** (CAR, BIKE, PLANE...)?
2. **P2 – Valor × desempenho:** existe relação entre o **valor monetário** e as métricas de desempenho? Quais classes/fabricantes oferecem a melhor relação desempenho/valor?
3. **P3 – Física × desempenho:** atributos de *handling* (massa, força do motor, velocidade de projeto, nº de marchas, força de freio) explicam as métricas de desempenho?
4. **P4 – Fabricantes:** quais fabricantes concentram os veículos mais rápidos e mais caros?
5. **P5 – Evolução por DLC (*power creep*):** veículos adicionados por DLCs têm desempenho e valor superiores aos do jogo base (`TitleUpdate`)?

## 2. Fontes de dados

| Item | `vehicles.json` | `vehicleHandlings.json` |
|---|---|---|
| **Origem** | Repositório público [loopgazer/GTA.V.Dataset](https://github.com/loopgazer/GTA.V.Dataset) (fork de DurtyFree/gta-v-data-dumps) — dados extraídos automaticamente dos arquivos do jogo, versão **v2944 (Online 1.67)** | mesmo repositório |
| **Commit fixado** | `8ff8268bac0d86a7e384f8b66d6b8dc5dd7c1eea` (21/06/2023) | idem |
| **Formato** | JSON (array de objetos, aninhado) | JSON (array de objetos, aninhado) |
| **Registros** | 815 veículos | 747 perfis de *handling* |
| **Granularidade** | 1 linha = 1 modelo de veículo | 1 linha = 1 perfil de dirigibilidade (compartilhável entre veículos) |
| **Principais atributos** | `Name`, `DisplayName`, `Manufacturer`, `Class`, `Type`, `DlcName`, `HandlingId`, `Seats`, `MonetaryValue`, `Price`, `MaxSpeed`, `Acceleration`, `MaxBraking`, `MaxTraction`, `Agility`, `Flags`, dimensões | `Id`, `Mass`, `InitialDriveForce`, `InitialDriveMaxFlatVel`, `InitialDriveGears`, `DriveBiasFront`, `BrakeForce`, `TractionCurveMax/Min`, `SuspensionForce`, multiplicadores de dano |
| **Relação com o problema** | Fornece categorias (P1, P4, P5), valor (P2) e métricas de desempenho (P1–P5) | Fornece as variáveis físicas que explicam o desempenho (P3) |

**Chave de ligação:** `vehicles.HandlingId` ↔ `vehicleHandlings.Id` (relação N:1 — vários veículos podem compartilhar o mesmo *handling*).

Os demais arquivos do repositório (sons, animações, cutscenes etc.) são listas de referência para *modding*, sem atributos analíticos relevantes ao problema, e por isso ficaram fora do escopo.

## 3. Ingestão dos dados

### 3.1 Ambiente
O Colab já traz o PySpark instalado; a célula abaixo só instala caso não esteja disponível (ex.: execução local).

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.3"])

In [ ]:
import hashlib
import json
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("RA1-GTAV-Veiculos")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| Python", sys.version.split()[0])

### 3.2 Parâmetros da ingestão
A URL aponta para um **commit fixo**, não para o branch: se o repositório de origem for atualizado, o notebook continua lendo exatamente os mesmos dados (reprodutibilidade).

In [ ]:
REPO = "loopgazer/GTA.V.Dataset"
COMMIT = "8ff8268bac0d86a7e384f8b66d6b8dc5dd7c1eea"
ARQUIVOS = ["vehicles.json", "vehicleHandlings.json"]

BASE = Path("/content/ra1") if Path("/content").exists() else Path("ra1")
RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
print("Dados brutos em:", RAW.resolve())

### 3.3 Download dos arquivos brutos (camada *raw*)
Cada arquivo é baixado uma vez, e registramos tamanho e hash SHA-256 para provar que os dados não mudaram entre execuções.

In [ ]:
def baixar(arquivo):
    url = f"https://raw.githubusercontent.com/{REPO}/{COMMIT}/{arquivo}"
    destino = RAW / arquivo
    if not destino.exists():
        urllib.request.urlretrieve(url, destino)
    return {
        "arquivo": arquivo,
        "url": url,
        "bytes": destino.stat().st_size,
        "sha256": hashlib.sha256(destino.read_bytes()).hexdigest(),
    }


log_ingestao = {
    "executado_em": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "repositorio": REPO,
    "commit": COMMIT,
    "arquivos": [baixar(a) for a in ARQUIVOS],
}
(RAW / "ingestao_log.json").write_text(json.dumps(log_ingestao, indent=2), encoding="utf-8")

for a in log_ingestao["arquivos"]:
    print(f"{a['arquivo']:<24} {a['bytes']:>10,} bytes  sha256={a['sha256'][:16]}...")

### 3.4 Leitura com PySpark
Os arquivos são um **array JSON multilinha**, por isso `multiLine=True`. O modo `FAILFAST` faz a leitura falhar caso exista algum registro malformado (em vez de gerar linhas corrompidas silenciosamente).
A coluna técnica `_ordem_origem` guarda a posição do registro no arquivo original (cada arquivo vira uma única partição, então a ordem é preservada) — ela será usada no tratamento de duplicatas.

In [ ]:
def ler_json(arquivo):
    return (
        spark.read
        .option("multiLine", True)
        .option("mode", "FAILFAST")
        .json(str(RAW / arquivo))
        .withColumn("_ordem_origem", F.monotonically_increasing_id())
    )


veh_raw = ler_json("vehicles.json").cache()
hdl_raw = ler_json("vehicleHandlings.json").cache()

print(f"vehicles.json         -> {veh_raw.count():>4} linhas x {len(veh_raw.columns) - 1} colunas (+ _ordem_origem)")
print(f"vehicleHandlings.json -> {hdl_raw.count():>4} linhas x {len(hdl_raw.columns) - 1} colunas (+ _ordem_origem)")

## 4. Compreensão e avaliação da qualidade dos dados

Funções auxiliares usadas nas análises:

In [ ]:
def tabela(linhas, colunas, n=100):
    # Exibe uma lista de tuplas como DataFrame Spark.
    if not linhas:
        print("(nenhum registro)")
        return
    spark.createDataFrame(linhas, colunas).show(n, truncate=False)


def tipos(df):
    return [(f.name, f.dataType.simpleString()[:70]) for f in df.schema.fields if not f.name.startswith("_")]


def perfil_ausentes(df):
    # Conta nulos; para strings, conta também valores vazios (""), que são ausências disfarçadas.
    exprs = []
    for f in df.schema.fields:
        if f.name.startswith("_"):
            continue
        cond = F.col(f.name).isNull()
        if isinstance(f.dataType, T.StringType):
            cond = cond | (F.trim(F.col(f.name)) == "")
        exprs.append(F.sum(F.when(cond, 1).otherwise(0)).alias(f.name))
    total = df.count()
    contagem = df.agg(*exprs).first().asDict()
    return [(c, int(n), round(100 * n / total, 1)) for c, n in contagem.items() if n > 0]

### 4.1 Estrutura e tipos
O Spark inferiu os tipos a partir do JSON. Verificamos se os numéricos vieram como número e quais colunas são aninhadas (struct/array) e precisarão ser achatadas.

In [ ]:
print("== vehicles.json ==")
tabela(tipos(veh_raw), ["coluna", "tipo"])

In [ ]:
print("== vehicleHandlings.json ==")
tabela(tipos(hdl_raw), ["coluna", "tipo"])

In [ ]:
veh_raw.select(
    "Name", F.col("DisplayName.English").alias("DisplayName_EN"), "Manufacturer", "Class", "Type",
    "DlcName", "HandlingId", "Price", "MonetaryValue", "MaxSpeed", "Acceleration", "MaxBraking"
).show(5, truncate=False)

**Observações:**
- Os tipos numéricos foram inferidos corretamente (`bigint`/`double`) — **não há problema de tipo** a corrigir na leitura.
- Há muitas colunas **aninhadas**: `DisplayName` e `ManufacturerDisplayName` (struct com o nome em 15 idiomas), `DimensionsMin/Max` (struct X/Y/Z), `Flags`, `Bones`, `DefaultColors`, `Extras` (arrays). Elas precisarão ser **achatadas** e só o que é relevante será mantido.
- Diversas colunas são técnicas do motor do jogo (`Hash`, `SignedHash`, `HexHash`, `Bones`, `LayoutId`, `DefaultHorn`...) e não têm relação com as perguntas analíticas.

### 4.2 Valores ausentes (nulos e strings vazias)

In [ ]:
print("== vehicles.json ==")
tabela(perfil_ausentes(veh_raw), ["coluna", "ausentes", "%"])
print("Nome em inglês (DisplayName.English) nulo:", veh_raw.filter(F.col("DisplayName.English").isNull()).count())

In [ ]:
veh_raw.filter(F.trim("Manufacturer") == "").groupBy("Type", "Class").count().orderBy(F.desc("count")).show(10)
veh_raw.filter(F.col("DisplayName.English").isNull()).select("Name", "Type", "DisplayName.Name").show()

In [ ]:
print("== vehicleHandlings.json ==")
tabela(perfil_ausentes(hdl_raw), ["coluna", "ausentes", "%"])

**Observações:**
- `Manufacturer` vem como **string vazia** (não nula) em 149 veículos — uma ausência "disfarçada", que uma contagem simples de nulos não detectaria. São majoritariamente veículos utilitários, aeronaves, emergência, trens e bicicletas: veículos **genéricos, sem marca no jogo**.
- 4 veículos (todos *trailers*) não têm nome de exibição em inglês.
- `Rewards`, `TranslatedDescription`, `SubHandlingData`, `AiHandling` etc. são quase sempre nulos, mas não são relevantes para o problema e serão descartados.

### 4.3 Valores sentinela, inválidos ou impossíveis
Ausências também podem estar codificadas como números (ex.: `-1`, `0`, `100`).

In [ ]:
print("Distribuição de Price:")
veh_raw.groupBy("Price").count().orderBy(F.desc("count")).show(6)

In [ ]:
metricas = ["MaxSpeed", "Acceleration", "MaxBraking", "MaxBrakingMods", "MaxTraction", "Agility", "MonetaryValue"]
tabela(
    [(m, veh_raw.filter(F.col(m) < 0).count(), veh_raw.filter(F.col(m) == 0).count()) for m in metricas],
    ["coluna", "negativos", "zeros"],
)

In [ ]:
print("Veículos com métricas negativas (por tipo):")
veh_raw.filter(F.col("MaxSpeed") < 0).groupBy("Type").count().show()

print("Carros (não-trailers) com métricas negativas:")
(veh_raw.filter((F.col("MaxSpeed") < 0) & (F.col("Type") != "TRAILER"))
    .select("Name", "Type", "DlcName", "MaxSpeed", "Acceleration", "MaxBraking", "Agility", "MonetaryValue")
    .show(truncate=False))

**Observações:**
- **`Price` não é um preço real:** 493 veículos têm `Price = 100` e 314 têm `Price = -1` (≈ 99% do total); só 8 veículos têm um valor diferente. A coluna é, na prática, um *placeholder* do jogo. O valor econômico real do veículo está em **`MonetaryValue`**.
- **`-1` é um valor sentinela** em todas as métricas de desempenho: velocidade, aceleração e frenagem negativas são fisicamente impossíveis.
  - 25 casos são **trailers** (reboques não têm motor próprio, logo não têm desempenho).
  - 5 casos são carros do DLC **`mpg9ec`** (versões exclusivas da edição *Expanded & Enhanced* para PS5/Xbox Series) — o extrator não obteve as estatísticas deles; os mesmos 5 têm `MonetaryValue = 0`.
- `MaxTraction = 0` em 26 veículos não-trailer é **legítimo** (aeronaves/barcos não dependem de tração no solo), assim como `MaxKnots`/`MoveResistance` = 0 fora de barcos/aeronaves.

### 4.4 Registros duplicados

In [ ]:
total = veh_raw.count()
print("vehicles: linhas =", total,
      "| Name distintos =", veh_raw.select("Name").distinct().count(),
      "| lower(Name) distintos =", veh_raw.select(F.lower("Name")).distinct().count())

print("\nNomes de exibição repetidos (variantes do mesmo modelo):")
(veh_raw.groupBy(F.col("DisplayName.English").alias("display_name"))
    .agg(F.count("*").alias("qtd"), F.collect_list("Name").alias("modelos"))
    .filter("qtd > 1").orderBy(F.desc("qtd")).show(8, truncate=False))

In [ ]:
hdl_dup = hdl_raw.groupBy(F.upper("Id").alias("id")).count().filter("count > 1")
hdl_sem_tecnica = hdl_raw.drop("_ordem_origem")
conflitantes = (hdl_sem_tecnica.distinct()
                .groupBy(F.upper("Id").alias("id")).count().filter("count > 1"))

print("handlings: linhas =", hdl_raw.count(),
      "| Id distintos =", hdl_raw.select(F.upper("Id")).distinct().count())
print("Ids repetidos:", hdl_dup.count())
print("Linhas que são cópia exata de outra:", hdl_raw.count() - hdl_sem_tecnica.distinct().count())
print("Ids repetidos com valores DIFERENTES entre as cópias:", conflitantes.count())
conflitantes.show(truncate=False)

**Observações:**
- Em `vehicles` **não há duplicatas**: `Name` é único (inclusive ignorando maiúsculas/minúsculas). Os 48 nomes de exibição repetidos são **variantes legítimas** do mesmo modelo (ex.: `boxville`, `boxville2`... são versões diferentes do mesmo furgão, com atributos distintos) — **não serão removidos**; a chave do conjunto será `Name`.
- Em `vehicleHandlings` **há duplicatas**: 24 `Id` aparecem duas vezes. 19 são **cópias exatas**; 5 têm pequenas diferenças em parâmetros de suspensão/tração (o mesmo *handling* foi redefinido por um DLC posterior). Isso, se não tratado, **duplicaria veículos no join**.

### 4.5 Formatos inconsistentes

In [ ]:
print("Name com letras maiúsculas:", veh_raw.filter(F.col("Name") != F.lower("Name")).count())
veh_raw.filter(F.col("Name") != F.lower("Name")).select("Name").show(8)

In [ ]:
ids_exatos = hdl_raw.select(F.col("Id").alias("hid")).distinct()
ids_upper = hdl_raw.select(F.upper(F.trim("Id")).alias("hid")).distinct()

sem_match_exato = veh_raw.join(ids_exatos, F.col("HandlingId") == F.col("hid"), "left_anti")
sem_match_upper = veh_raw.join(ids_upper, F.upper(F.trim("HandlingId")) == F.col("hid"), "left_anti")
print("Veículos sem handling (comparação exata)        :", sem_match_exato.count())
print("Veículos sem handling (após padronizar caixa)   :", sem_match_upper.count())

(sem_match_exato.select("Name", "HandlingId").limit(5)
    .join(hdl_raw.select("Id"), F.upper("HandlingId") == F.upper("Id")).show())

In [ ]:
print("Mesmo fabricante escrito com códigos diferentes:")
(veh_raw.filter(F.trim("Manufacturer") != "")
    .groupBy(F.col("ManufacturerDisplayName.English").alias("fabricante"))
    .agg(F.collect_set("Manufacturer").alias("codigos"), F.count("*").alias("veiculos"))
    .filter(F.size("codigos") > 1).show(truncate=False))

**Observações:**
- **Caixa inconsistente em chaves:** 94 valores de `Name` têm letras maiúsculas (`Baller`, `BARRACKS`, `Burrito`...) enquanto a maioria é minúscula. O mesmo ocorre entre `HandlingId` e `Id`: **78 veículos não encontram seu *handling*** numa comparação exata (ex.: `ALPHA` × `Alpha`) — e **todos** passam a casar após padronizar para maiúsculas.
- **Códigos de fabricante truncados:** `BENEFAC`/`BENEFACTOR`, `LAMPADA`/`LAMPADATI` e `UBERMACH`/`UBERMACHT` representam o mesmo fabricante. Agrupar por `Manufacturer` dividiria esses fabricantes em dois. O nome de exibição (`ManufacturerDisplayName.English`) é consistente e será usado.

### 4.6 Consistência das categorias

In [ ]:
veh_raw.groupBy("Type").count().orderBy(F.desc("count")).show(20)
veh_raw.groupBy("Class", "ClassId").count().orderBy("ClassId").show(30)

In [ ]:
for c in ["Type", "Class", "DlcName"]:
    fora_padrao = veh_raw.filter(F.col(c) != F.trim(F.col(c))).count()
    print(f"{c:<8}: {veh_raw.select(c).distinct().count():>2} categorias | com espaços extras: {fora_padrao}")
print("Class x ClassId 1:1 ->", veh_raw.select("Class", "ClassId").distinct().count() == veh_raw.select("Class").distinct().count())

**Observações:** as categorias de `Type`, `Class` e `DlcName` são **consistentes** (sem variações de grafia, espaços extras ou caixa misturada) e `Class` ↔ `ClassId` é 1:1. **Não há tratamento necessário** aqui — a única exceção é a categoria `TRAILER`, discutida na seção 5.

### 4.7 Possíveis outliers
Usamos o critério do **IQR** (valores fora de `[Q1 − 1,5·IQR ; Q3 + 1,5·IQR]`), primeiro no conjunto inteiro e depois **dentro de cada `Type`** — porque comparar a velocidade de um avião com a de uma bicicleta não faz sentido. Os sentinelas (`< 0`) são excluídos do cálculo.

In [ ]:
METRICAS_OUTLIER = ["MaxSpeed", "Acceleration", "MonetaryValue"]
validos = veh_raw.filter((F.col("MaxSpeed") > 0) & (F.col("MonetaryValue") > 0))


def contar_outliers(df, col, por=None):
    chaves = [por] if por else []
    q = df.groupBy(*chaves).agg(F.percentile_approx(col, [0.25, 0.75]).alias("q"))
    q = q.select(*chaves, F.col("q")[0].alias("q1"), F.col("q")[1].alias("q3"))
    j = df.crossJoin(q) if not por else df.join(q, por)
    iqr = F.col("q3") - F.col("q1")
    return j.filter((F.col(col) < F.col("q1") - 1.5 * iqr) | (F.col(col) > F.col("q3") + 1.5 * iqr))


tabela(
    [(m, contar_outliers(validos, m).count(), contar_outliers(validos, m, "Type").count()) for m in METRICAS_OUTLIER],
    ["metrica", "outliers_global", "outliers_dentro_do_Type"],
)

In [ ]:
print("Resumo por tipo (MaxSpeed em m/s):")
(validos.groupBy("Type")
    .agg(F.count("*").alias("n"),
         F.round(F.min("MaxSpeed"), 1).alias("min_speed"),
         F.round(F.max("MaxSpeed"), 1).alias("max_speed"),
         F.round(F.max("Acceleration"), 2).alias("max_accel"),
         F.max("MonetaryValue").alias("max_valor"))
    .orderBy(F.desc("max_speed")).show(20))

print("Maiores outliers de MaxSpeed dentro de CAR:")
(contar_outliers(validos, "MaxSpeed", "Type").filter("Type = 'CAR'")
    .select("Name", "Class", "MaxSpeed", "Acceleration", "MonetaryValue").orderBy(F.desc("MaxSpeed")).show(10))

**Observações:** a maior parte dos "outliers globais" desaparece quando a análise é feita **por tipo**: aviões chegam a ~110 m/s e têm aceleração 20× maior que a dos carros por serem outra categoria, não por erro. Os outliers que restam dentro de cada tipo (ex.: carros de corrida/*open wheel* muito rápidos, veículos militares muito caros) são **valores plausíveis do jogo**, não erros de medição.

### 4.8 Integridade referencial entre as fontes

In [ ]:
handling_usados = veh_raw.select(F.upper(F.trim("HandlingId")).alias("hid")).distinct()
orfaos = ids_upper.join(handling_usados, "hid", "left_anti")
print("Handlings não usados por nenhum veículo:", orfaos.count())
orfaos.show(truncate=False)

compartilhados = veh_raw.groupBy(F.upper("HandlingId").alias("hid")).count().filter("count > 1")
print("Handlings compartilhados por mais de um veículo:", compartilhados.count())

**Observações:** 12 *handlings* não pertencem a nenhum veículo do catálogo (sobras de versões antigas/veículos removidos) — serão naturalmente descartados por um *left join* a partir dos veículos. O compartilhamento de *handling* entre veículos (76 casos) é **esperado** (variantes usam a mesma física) e confirma a relação N:1.

### 4.9 Resumo dos problemas de qualidade

| # | Problema | Onde | Volume | Relevância |
|---|---|---|---|---|
| Q1 | Ausência disfarçada (string vazia) | `vehicles.Manufacturer` | 149 (18%) | Alta – afeta P4 |
| Q2 | Nome de exibição ausente | `DisplayName.English` | 4 | Baixa |
| Q3 | Coluna sem informação real (placeholder `100`/`-1`) | `vehicles.Price` | 807 (99%) | Alta – invalidaria P2 se usada |
| Q4 | Sentinela `-1` em métricas de desempenho / `0` em valor | `MaxSpeed`, `Acceleration`, `MaxBraking`, `Agility`, `MonetaryValue` | 30 (25 trailers + 5 carros `mpg9ec`) | Alta – distorce médias de P1–P5 |
| Q5 | Duplicatas (exatas e conflitantes) | `vehicleHandlings.Id` | 24 Ids | Alta – duplicaria linhas no join |
| Q6 | Caixa inconsistente em chaves | `Name`, `HandlingId` × `Id` | 94 / 78 joins perdidos | Alta – perderia 10% dos veículos no join |
| Q7 | Códigos de categoria truncados | `vehicles.Manufacturer` | 3 fabricantes | Média – afeta P4 |
| Q8 | Estruturas aninhadas e colunas técnicas | ambos | ~40 colunas | Média – dificulta uso |
| Q9 | Outliers | métricas | explicados pelo `Type` | Baixa – não são erros |
| Q10 | Registros fora do escopo (sem motor) | `Type = TRAILER` | 25 | Média |
| – | Tipos de dados / categorias `Type`/`Class` | ambos | sem problemas | – |

## 5. Limpeza e preparação dos dados

Cada passo abaixo referencia o problema (Q#) da seção 4.9 e a justificativa da decisão.

### 5.1 `vehicleHandlings`: padronizar chave e remover duplicatas (Q5, Q6, Q8)
- **Chave padronizada** para maiúsculas sem espaços (`handling_id`), permitindo o join (Q6).
- **Deduplicação:** para cada `handling_id`, mantemos a **última ocorrência no arquivo** (`_ordem_origem` mais alto). Para as 19 cópias exatas a escolha é indiferente; para as 5 conflitantes, assumimos que a definição posterior corresponde à redefinição feita por um DLC mais recente, que é a que vale no jogo. As diferenças são pequenas (parâmetros de suspensão/tração) e não afetam as variáveis principais.
- **Seleção de colunas:** mantemos apenas os atributos físicos ligados à pergunta P3, com nomes em *snake_case* e unidade explícita quando conhecida. `MonetaryValue` do *handling* é descartado (o valor oficial é o do veículo).

In [ ]:
ultima_ocorrencia = Window.partitionBy("handling_id").orderBy(F.desc("_ordem_origem"))

hdl = (
    hdl_raw
    .withColumn("handling_id", F.upper(F.trim("Id")))
    .withColumn("_rn", F.row_number().over(ultima_ocorrencia))
    .filter("_rn = 1")
    .select(
        "handling_id",
        F.col("Mass").alias("mass_kg"),
        F.col("InitialDragCoeff").alias("drag_coeff"),
        F.col("DriveBiasFront").alias("drive_bias_front"),
        F.col("InitialDriveGears").cast("int").alias("drive_gears"),
        F.col("InitialDriveForce").alias("drive_force"),
        F.col("InitialDriveMaxFlatVel").alias("drive_max_flat_vel"),
        F.col("BrakeForce").alias("brake_force"),
        F.col("HandBrakeForce").alias("handbrake_force"),
        F.col("SteeringLock").alias("steering_lock_deg"),
        F.col("TractionCurveMax").alias("traction_curve_max"),
        F.col("TractionCurveMin").alias("traction_curve_min"),
        F.col("SuspensionForce").alias("suspension_force"),
        F.col("CollisionDamageMult").alias("collision_damage_mult"),
        F.col("DeformationDamageMult").alias("deformation_damage_mult"),
        F.col("EngineDamageMult").alias("engine_damage_mult"),
        F.col("PetrolTankVolume").alias("petrol_tank_volume"),
    )
)
print("handlings: antes =", hdl_raw.count(), "| depois =", hdl.count(),
      "| handling_id único:", hdl.count() == hdl.select("handling_id").distinct().count())

### 5.2 `vehicles`: escopo, achatamento, padronização e sentinelas
| Decisão | Problema | Justificativa |
|---|---|---|
| Remover `Type = TRAILER` | Q10, Q4 | Reboques não têm propulsão própria: todas as métricas de desempenho são `-1`. Não respondem a nenhuma pergunta sobre desempenho e distorceriam as estatísticas. |
| `name` em minúsculas; `handling_id` em maiúsculas | Q6 | Chaves com caixa única; garante o join e evita duplicidade lógica. |
| `display_name` = `DisplayName.English`; se nulo, usar `Name` | Q2, Q8 | Achatar o struct multilíngue mantendo só o idioma de análise; *fallback* evita nulos (após remover trailers, não deve sobrar nenhum). |
| `manufacturer` = `ManufacturerDisplayName.English`; vazio → `"Sem fabricante"` | Q1, Q7 | O nome de exibição unifica códigos truncados (`BENEFAC`/`BENEFACTOR`). A ausência não é desconhecimento: são veículos **genéricos sem marca** no jogo, então uma categoria explícita é mais fiel que nulo e evita perder linhas em agrupamentos. |
| Remover `Price` | Q3 | 99% *placeholder*; manter a coluna induziria análises erradas. `monetary_value` é a métrica de valor. |
| Métricas negativas → `null`; `MonetaryValue = 0` → `null` | Q4 | `-1` e `0` são sentinelas de "dado não extraído". Imputar (média/mediana) inventaria desempenho para 5 carros específicos; removê-los perderia veículos válidos do catálogo. Mantemos a linha com nulo e uma *flag* `has_performance_stats`. |
| `max_speed_kmh` = `MaxSpeed × 3,6` | padronização | `MaxSpeed` está em m/s (unidade interna do jogo); km/h facilita a interpretação. A coluna original em m/s é mantida. |
| Dimensões → `length_m`, `width_m`, `height_m` | Q8 | Achatar os structs X/Y/Z em medidas diretas (max − min). |
| Descartar colunas técnicas (`Hash*`, `Bones`, `DefaultColors`, `Extras`, `ModKits`, `LayoutId`, `DefaultHorn`, `MaxKnots`, `MoveResistance`...) | Q8 | Não se relacionam ao problema; `MaxKnots`/`MoveResistance` só têm valor para barcos/aeronaves. |
| **Manter** outliers | Q9 | São explicados pelo tipo de veículo e plausíveis no jogo; removê-los eliminaria justamente os veículos de alto desempenho que interessam a P1/P2. Nos próximos RAs as comparações devem ser feitas **dentro de cada `vehicle_type`**. |

In [ ]:
def nao_negativo(col):
    # Sentinela: valores < 0 são impossíveis para estas métricas -> null.
    return F.when(F.col(col) >= 0, F.col(col))


veh = (
    veh_raw
    .filter(F.col("Type") != "TRAILER")
    .select(
        F.lower(F.trim("Name")).alias("name"),
        F.coalesce(F.col("DisplayName.English"), F.col("Name")).alias("display_name"),
        F.coalesce(
            F.when(F.trim("Manufacturer") != "", F.col("ManufacturerDisplayName.English")),
            F.lit("Sem fabricante"),
        ).alias("manufacturer"),
        F.col("Type").alias("vehicle_type"),
        F.col("Class").alias("vehicle_class"),
        F.col("ClassId").cast("int").alias("class_id"),
        F.col("DlcName").alias("dlc_name"),
        F.upper(F.trim("HandlingId")).alias("handling_id"),
        F.col("Seats").cast("int").alias("seats"),
        F.col("WheelsCount").cast("int").alias("wheels_count"),
        F.when(F.col("MonetaryValue") > 0, F.col("MonetaryValue")).cast("long").alias("monetary_value"),
        nao_negativo("MaxSpeed").alias("max_speed_ms"),
        F.round(nao_negativo("MaxSpeed") * 3.6, 2).alias("max_speed_kmh"),
        nao_negativo("Acceleration").alias("acceleration"),
        nao_negativo("MaxBraking").alias("max_braking"),
        nao_negativo("MaxBrakingMods").alias("max_braking_mods"),
        nao_negativo("MaxTraction").alias("max_traction"),
        nao_negativo("Agility").alias("agility"),
        (F.col("MaxSpeed") >= 0).alias("has_performance_stats"),
        F.col("DefaultBodyHealth").alias("default_body_health"),
        F.col("SpawnFrequency").alias("spawn_frequency"),
        F.round(F.col("DimensionsMax.Y") - F.col("DimensionsMin.Y"), 3).alias("length_m"),
        F.round(F.col("DimensionsMax.X") - F.col("DimensionsMin.X"), 3).alias("width_m"),
        F.round(F.col("DimensionsMax.Z") - F.col("DimensionsMin.Z"), 3).alias("height_m"),
        F.col("HasConvertibleRoof").alias("has_convertible_roof"),
        F.col("HasSirens").alias("has_sirens"),
        F.col("HasArmoredWindows").alias("has_armored_windows"),
        F.col("HasParachute").alias("has_parachute"),
        F.col("HasKers").alias("has_kers"),
        (F.size("Weapons") > 0).alias("has_weapons"),
        F.coalesce("Flags", F.array().cast("array<string>")).alias("flags"),
    )
)
print("vehicles: antes =", veh_raw.count(), "| depois de remover trailers =", veh.count())

### 5.3 Integração `vehicles` + `vehicleHandlings`
*Left join* a partir dos veículos (o veículo é a unidade de análise; *handlings* órfãos ficam de fora). Com as chaves padronizadas e os *handlings* deduplicados, esperamos **nenhum veículo sem par** e **nenhuma linha duplicada**.

In [ ]:
vehicles_clean = veh.join(hdl, "handling_id", "left")

sem_handling = vehicles_clean.filter(F.col("mass_kg").isNull()).count()
print("linhas antes do join:", veh.count(), "| depois:", vehicles_clean.count(), "| sem handling:", sem_handling)

colunas_finais = ["name", "display_name", "manufacturer", "vehicle_type", "vehicle_class", "class_id", "dlc_name",
                  "handling_id"] + [c for c in vehicles_clean.columns if c not in
                  ("name", "display_name", "manufacturer", "vehicle_type", "vehicle_class", "class_id", "dlc_name", "handling_id")]
vehicles_clean = vehicles_clean.select(*colunas_finais).orderBy("name").cache()
vehicles_clean.printSchema()

## 6. Conjunto de dados resultante do RA1

### 6.1 Validações automáticas
Cada regra abaixo verifica que um problema da seção 4.9 foi de fato resolvido. Se alguma falhar, a célula interrompe a execução.

In [ ]:
df = vehicles_clean
metricas_finais = ["max_speed_ms", "acceleration", "max_braking", "max_traction", "agility", "monetary_value"]

checks = {
    "name é chave única (sem duplicatas)": df.count() == df.select("name").distinct().count(),
    "name sem nulos e em minúsculas (Q6)": df.filter(F.col("name").isNull() | (F.col("name") != F.lower("name"))).count() == 0,
    "join não duplicou linhas (Q5)": df.count() == veh.count(),
    "todo veículo tem handling (Q6)": df.filter(F.col("mass_kg").isNull()).count() == 0,
    "manufacturer sem nulos/vazios (Q1)": df.filter(F.col("manufacturer").isNull() | (F.trim("manufacturer") == "")).count() == 0,
    "códigos truncados unificados (Q7)": df.filter(F.upper("manufacturer").isin("BENEFAC", "LAMPADA", "UBERMACH")).count() == 0,
    "display_name sem nulos (Q2)": df.filter(F.col("display_name").isNull()).count() == 0,
    "sem sentinelas negativos/zero nas métricas (Q4)": df.filter(" OR ".join(f"{m} < 0" for m in metricas_finais) + " OR monetary_value = 0").count() == 0,
    "nulos de desempenho só onde has_performance_stats = false (Q4)": df.filter(F.col("max_speed_ms").isNull() & F.col("has_performance_stats")).count() == 0,
    "sem trailers (Q10)": df.filter(F.col("vehicle_type") == "TRAILER").count() == 0,
    "coluna Price removida (Q3)": "price" not in [c.lower() for c in df.columns],
}
for regra, ok in checks.items():
    print("OK    " if ok else "FALHOU", regra)
assert all(checks.values()), "Alguma validação falhou"

### 6.2 Comparativo de qualidade: antes × depois

In [ ]:
antes = {
    "linhas (vehicles)": veh_raw.count(),
    "fabricante vazio": veh_raw.filter(F.trim("Manufacturer") == "").count(),
    "códigos de fabricante distintos": veh_raw.select("Manufacturer").distinct().count(),
    "métricas com sentinela (<0)": veh_raw.filter(F.col("MaxSpeed") < 0).count(),
    "valor monetário = 0": veh_raw.filter(F.col("MonetaryValue") == 0).count(),
    "veículos sem handling no join": sem_match_exato.count(),
    "handlings duplicados": hdl_raw.count() - hdl_raw.select(F.upper("Id")).distinct().count(),
    "colunas": len(veh_raw.columns) - 1,
}
depois = {
    "linhas (vehicles)": df.count(),
    "fabricante vazio": df.filter(F.col("manufacturer").isNull() | (F.trim("manufacturer") == "")).count(),
    "códigos de fabricante distintos": df.select("manufacturer").distinct().count(),
    "métricas com sentinela (<0)": df.filter(F.col("max_speed_ms") < 0).count(),
    "valor monetário = 0": df.filter(F.col("monetary_value") == 0).count(),
    "veículos sem handling no join": df.filter(F.col("mass_kg").isNull()).count(),
    "handlings duplicados": hdl.count() - hdl.select("handling_id").distinct().count(),
    "colunas": len(df.columns),
}
tabela([(k, int(antes[k]), int(depois[k])) for k in antes], ["indicador", "antes", "depois"])
print("Veículos mantidos com métricas nulas (sem estatísticas no jogo):",
      [r["name"] for r in df.filter(~F.col("has_performance_stats")).select("name").collect()])

### 6.3 Amostra e resumo do conjunto final

In [ ]:
df.select("name", "display_name", "manufacturer", "vehicle_type", "vehicle_class", "dlc_name",
          "monetary_value", "max_speed_kmh", "acceleration", "max_braking", "mass_kg", "drive_gears").show(10, truncate=False)
df.select("monetary_value", "max_speed_kmh", "acceleration", "max_braking", "mass_kg", "drive_force").summary(
    "count", "mean", "min", "50%", "max").show()
tabela(perfil_ausentes(df), ["coluna", "ausentes", "%"])

### 6.4 Dicionário de dados

| Coluna | Tipo | Descrição |
|---|---|---|
| `name` | string | **Chave.** Nome do modelo do veículo no jogo (minúsculas) |
| `display_name` | string | Nome exibido ao jogador (inglês) |
| `manufacturer` | string | Fabricante (nome de exibição); `"Sem fabricante"` para veículos genéricos |
| `vehicle_type` | string | Tipo físico: CAR, BIKE, PLANE, HELI, BOAT... |
| `vehicle_class` / `class_id` | string / int | Classe do jogo: SUPER, SPORT, MUSCLE... |
| `dlc_name` | string | Pacote em que o veículo foi adicionado (`TitleUpdate` = jogo base) |
| `handling_id` | string | Chave do perfil de dirigibilidade (maiúsculas) |
| `seats`, `wheels_count` | int | Assentos e rodas |
| `monetary_value` | long | Valor monetário do veículo no jogo (US$ do jogo); nulo se não extraído |
| `max_speed_ms` / `max_speed_kmh` | double | Velocidade máxima estimada (m/s e km/h) |
| `acceleration`, `max_braking`, `max_braking_mods`, `max_traction`, `agility` | double | Métricas de desempenho (escala interna do jogo) |
| `has_performance_stats` | boolean | `false` quando o jogo não fornece as métricas (5 veículos `mpg9ec`) |
| `default_body_health`, `spawn_frequency` | double | Vida padrão da carroceria e frequência de aparição no trânsito |
| `length_m`, `width_m`, `height_m` | double | Dimensões da caixa delimitadora (m) |
| `has_*` | boolean | Teto conversível, sirene, vidros blindados, paraquedas, KERS, armas |
| `flags` | array&lt;string&gt; | Marcadores do jogo (ex.: `FLAG_SPORTS`, `FLAG_RICH_CAR`) |
| `mass_kg` | double | Massa (kg) — *handling* |
| `drag_coeff` | double | Coeficiente de arrasto — *handling* |
| `drive_bias_front` | double | Distribuição da tração (0 = traseira, 1 = dianteira, 0,5 = 4x4) |
| `drive_gears` | int | Número de marchas |
| `drive_force` | double | Força do motor |
| `drive_max_flat_vel` | double | Velocidade de projeto em pista plana |
| `brake_force`, `handbrake_force` | double | Força de freio e de freio de mão |
| `steering_lock_deg` | double | Ângulo máximo de esterçamento (graus) |
| `traction_curve_max/min` | double | Aderência máxima/mínima dos pneus |
| `suspension_force` | double | Rigidez da suspensão |
| `*_damage_mult` | double | Multiplicadores de dano (colisão, deformação, motor) |
| `petrol_tank_volume` | double | Volume do tanque |

### 6.5 Persistência (camada *processed*)
O conjunto é salvo em **Parquet** (preserva tipos e a coluna `flags` como lista) e em **CSV** (para inspeção rápida; `flags` vira texto separado por `|`).

In [ ]:
saida_parquet = PROCESSED / "vehicles_clean.parquet"
saida_csv = PROCESSED / "vehicles_clean.csv"

df.coalesce(1).write.mode("overwrite").parquet(str(saida_parquet))
df.withColumn("flags", F.concat_ws("|", "flags")).toPandas().to_csv(saida_csv, index=False)

relido = spark.read.parquet(str(saida_parquet))
assert relido.count() == df.count() and relido.columns == df.columns
print(f"Parquet: {saida_parquet}  ({relido.count()} linhas x {len(relido.columns)} colunas)")
print(f"CSV    : {saida_csv}")

### 6.6 (Opcional) Copiar resultados para o Google Drive / baixar
O disco do Colab é temporário. Mude `COPIAR_PARA_DRIVE` para `True` para guardar os arquivos no seu Drive.

In [ ]:
COPIAR_PARA_DRIVE = False

if COPIAR_PARA_DRIVE:
    import shutil
    from google.colab import drive

    drive.mount("/content/drive")
    destino_drive = Path("/content/drive/MyDrive/PjBL_RA1_GTAV")
    shutil.copytree(BASE / "data", destino_drive / "data", dirs_exist_ok=True)
    print("Copiado para", destino_drive)

## Conclusão do RA1

O conjunto **`vehicles_clean`** (790 veículos, 1 linha por modelo, chave `name`) integra os atributos de catálogo, desempenho e física (*handling*) e está pronto para o **RA2 – Transformação e Processamento**. Os principais problemas de qualidade (sentinelas, ausências disfarçadas, coluna *placeholder*, duplicatas de *handling*, inconsistência de caixa nas chaves e códigos de fabricante truncados) foram identificados, tratados e validados automaticamente.

**Pontos de atenção para os próximos RAs:**
- Comparar métricas **dentro de cada `vehicle_type`** (escalas muito diferentes entre carros e aeronaves).
- 5 veículos (`has_performance_stats = false`) devem ser excluídos de análises de desempenho, mas podem ser usados em contagens de catálogo.
- `dlc_name` é um código; para P5 será preciso associá-lo à data de lançamento de cada DLC.